<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Receta de la Plaza: Lavar, Pelar y Cocinar en Orden 🍲
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/06%20-%20Modelamiento%20Estadistico/Para%20Dummies/03_Receta_Pipeline_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 03 del Módulo 06 (Ejercicio práctico con scikit-learn y statsmodels)](../03_Ejercicio_Practico_Scikit_Learn_y_Statsmodels.ipynb). Piensa en preparar un **ajiaco**: primero **lavas** las papas, luego las **pelas**, las **cortas** y por último las **cocinas**, y el orden importa. Un modelo también necesita su receta: **rellenar los datos que faltan, poner las cifras en la misma escala, traducir las palabras a números y recién ahí ajustar el modelo**. Y como a toda receta, la haces **igual** con los ingredientes nuevos del día siguiente.

| En la cocina | En el modelo |
|---|---|
| Lavar | **Rellenar datos faltantes** (imputar) |
| Pelar y pesar | **Poner las cifras en la misma escala** |
| Cortar por tipo de ingrediente | **Traducir las palabras a casillas** (una receta distinta para papas y para hierbas) |
| Cocinar | **Ajustar el modelo** |
| Escribir la receta y repetirla tal cual | El ***pipeline*** (tubería): una sola receta que se aplica igual a datos nuevos |
| El catador que dice «sabe bien» | **scikit-learn**: ¿predice bien con datos nuevos? |
| El cocinero que explica «la sal aportó tanto, la papa tanto» | **statsmodels**: ¿cuánto aporta cada ingrediente, con qué margen de error? |

Al terminar podrás explicar:
1. Por qué conviene una **receta única** (*pipeline*) en vez de preparar los datos «a mano».
2. Cómo se **prueba la receta completa** con datos que no vio, sin hacer trampa.
3. La diferencia entre el cocinero que **explica** (statsmodels) y el catador que **predice** (scikit-learn), y por qué dan **los mismos números**.

> 🔗 Si luego quieres ver las versiones **con diagnóstico de supuestos, GLM y comparación de modelos**: [cuaderno estándar](../03_Ejercicio_Practico_Scikit_Learn_y_Statsmodels.ipynb). Taller del módulo: [`../../homeworks/Para Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb).

---
## 1. Una Sola Receta para Todo 🥘

Tenemos 300 fincas **inventadas** con: altitud, fertilizante (a veces **falta el dato**), variedad (a veces **falta**) y nivel de manejo, y queremos predecir el rendimiento. La receta (*pipeline*) hace en orden: (1) **rellenar** lo que falta con un valor típico; (2) **escalar** las cifras; (3) **traducir** la variedad a casillas de sí/no y el manejo a un orden (Bajo < Medio < Alto); (4) **ajustar** el modelo. Lo importante: **todo lo que la receta «aprende» (los valores típicos, las escalas) lo aprende solo de las fincas de estudio** de cada ronda, nunca de las fincas del examen. Y luego prueba la receta completa con **validación cruzada** (varios exámenes).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import statsmodels.formula.api as smf

rng = np.random.default_rng(11)
n = 300
fincas = pd.DataFrame({"altitud": rng.normal(2800, 150, n), "fert": rng.gamma(4, 50, n),
                       "variedad": rng.choice(["Capiro", "Criolla", "Pastusa"], n, p=[0.4, 0.2, 0.4]),
                       "manejo": rng.choice(["Bajo", "Medio", "Alto"], n)})
efecto_var = fincas["variedad"].map({"Capiro": 0, "Criolla": -3, "Pastusa": 4})
efecto_man = fincas["manejo"].map({"Bajo": 0, "Medio": 1.5, "Alto": 3})
fincas["rend"] = 15 + 0.02 * (fincas["fert"] - 200) + efecto_var + efecto_man + rng.normal(0, 2, n)      # INVENTADO
fincas.loc[rng.choice(n, 30, replace=False), "fert"] = np.nan                                           # faltan 30 datos de fertilizante
fincas.loc[rng.choice(n, 15, replace=False), "variedad"] = np.nan                                       # y 15 variedades
print("Datos que faltan:", fincas.isna().sum()[lambda s: s > 0].to_dict())

X, y = fincas.drop(columns="rend"), fincas["rend"]
receta = make_pipeline(
    ColumnTransformer([
        ("cifras", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["altitud", "fert"]),          # lavar y pesar
        ("palabras", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(drop="first", handle_unknown="ignore")), ["variedad"]),
        ("orden", OrdinalEncoder(categories=[["Bajo", "Medio", "Alto"]]), ["manejo"])]),                                # cortar
    LinearRegression())                                                                                                  # cocinar

notas = cross_val_score(receta, X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
print("R2 en cada uno de los 5 examenes:", notas.round(2), "| promedio =", notas.mean().round(2))
assert notas.mean() > 0.5 and (notas > 0.3).all()

# La receta ya entrenada se aplica IGUAL a una finca nueva, aunque traiga datos que faltan o una variedad nunca vista
receta.fit(X, y)
nueva = pd.DataFrame([{"altitud": 2850, "fert": np.nan, "variedad": "Tuquerreña", "manejo": "Alto"}])
print(f"\nFinca nueva con fertilizante faltante y variedad desconocida -> rendimiento predicho: {receta.predict(nueva)[0]:.1f} t/ha")
assert np.isfinite(receta.predict(nueva)[0])

---
**Lección:** la receta única rellena, escala, traduce y ajusta **en el orden correcto**, y funciona con una finca nueva aunque le falten datos. La validación cruzada la prueba **completa** (incluido el rellenado): ≈ 0.7 de R² promedio en 5 exámenes distintos. Si hubieras rellenado y escalado **antes** de partir los datos, el examen habría «espiado» a las fincas de estudio. Eso es hacer trampa sin querer.

---
## 2. Dos Cocineros: el que Explica y el que Predice 👩‍🍳

Con los mismos ingredientes, hay dos formas de trabajar. **scikit-learn** es el catador: te dice si el plato **sale bien** con ingredientes nuevos (predice). **statsmodels** es el cocinero que explica: te dice **cuánto aporta cada ingrediente** y con **qué margen de error** («un nivel más de manejo suma 1.6 t/ha, con margen de ± 0.3»). Si miran las **mismas fincas completas**, deberían dar **las mismas predicciones**: son dos puertas al mismo modelo.

In [ ]:
completas = fincas.dropna().copy()
completas["manejo_num"] = completas["manejo"].map({"Bajo": 0, "Medio": 1, "Alto": 2})

# El que EXPLICA: coeficientes con margen de error (intervalo de confianza)
cocinero = smf.ols("rend ~ altitud + fert + C(variedad) + manejo_num", completas).fit()
tabla = pd.concat([cocinero.params.rename("aporte"), cocinero.conf_int().rename(columns={0: "margen_inf", 1: "margen_sup"})], axis=1).round(2)
print(tabla.to_string())

# El que PREDICE: la receta de scikit-learn con las mismas columnas (sin rellenar nada: no falta nada)
catador = make_pipeline(ColumnTransformer([("c", "passthrough", ["altitud", "fert"]), ("p", OneHotEncoder(drop="first"), ["variedad"]),
                                           ("o", OrdinalEncoder(categories=[["Bajo", "Medio", "Alto"]]), ["manejo"])]), LinearRegression())
catador.fit(completas[["altitud", "fert", "variedad", "manejo"]], completas["rend"])
diferencia = np.abs(catador.predict(completas[["altitud", "fert", "variedad", "manejo"]]) - cocinero.fittedvalues).max()
print(f"\nDiferencia maxima entre las predicciones de los dos cocineros: {diferencia:.1e}")
assert diferencia < 1e-8
assert tabla.loc["C(variedad)[T.Pastusa]", "margen_inf"] > 0 > tabla.loc["C(variedad)[T.Criolla]", "margen_sup"]

---
**Lección:** los dos cocineros dan **exactamente las mismas predicciones** (diferencia ≈ 0), pero cada uno aporta algo distinto: statsmodels te dice, por ejemplo, que **Pastusa rinde más y Criolla menos que Capiro**, y con **qué margen** (los intervalos no incluyen el cero); scikit-learn te da la receta repetible y la forma honesta de **probarla** con exámenes. En la práctica se usan **juntos**: statsmodels para entender, scikit-learn para validar y poner en producción.

---
##### 🛠️ Práctica 1: Tu turno: ¿qué rinde esta finca?

Con la `receta` ya entrenada de la sección 1:

1. Crea `finca_a` con `altitud=2800`, `fert=250`, `variedad="Pastusa"`, `manejo="Alto"` y `finca_b` igual pero con `variedad="Criolla"` y `manejo="Bajo"`. Cada una es un `DataFrame` de una fila con las columnas de `X`.
2. Predice los dos rendimientos: `pred_a` y `pred_b`.
3. Verifica con `assert` que `pred_a > pred_b + 5` (según los datos inventados, la mejor combinación rinde mucho más que la peor). Comenta: ¿cuántas toneladas por hectárea de diferencia estima el modelo?

In [ ]:
# Escribe tu código aquí

# finca_a = pd.DataFrame([{"altitud": 2800, "fert": 250, "variedad": "Pastusa", "manejo": "Alto"}])
# finca_b = ...
# pred_a = receta.predict(finca_a)[0]


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
finca_a = pd.DataFrame([{"altitud": 2800, "fert": 250, "variedad": "Pastusa", "manejo": "Alto"}])
finca_b = pd.DataFrame([{"altitud": 2800, "fert": 250, "variedad": "Criolla", "manejo": "Bajo"}])
pred_a, pred_b = receta.predict(finca_a)[0], receta.predict(finca_b)[0]
print(f"Pastusa + manejo alto: {pred_a:.1f} t/ha | Criolla + manejo bajo: {pred_b:.1f} t/ha | diferencia = {pred_a - pred_b:.1f} t/ha")

assert pred_a > pred_b + 5
```
</details>

---


---
## Resumen en una frase 🎯

> **Un *pipeline* es una receta única (rellenar, escalar, traducir, ajustar) que se aprende solo con los datos de estudio y se aplica igual a los nuevos; statsmodels te dice cuánto aporta cada ingrediente y scikit-learn te dice si el plato sale bien con ingredientes nuevos, y ambos dan los mismos números.**

### Lo que aprendiste hoy:
- ✅ El **orden** de la receta importa: rellenar, escalar, traducir y recién ahí ajustar.
- ✅ Todo lo que la receta aprende de los datos se aprende **solo con las fincas de estudio** (así no hay trampa).
- ✅ La receta se prueba **completa** con varios exámenes (validación cruzada) y funciona con fincas nuevas, incluso con datos faltantes o variedades desconocidas.
- ✅ **statsmodels** explica (aportes con margen de error); **scikit-learn** predice y valida; con los mismos datos dan **las mismas predicciones**.

---
### 🧠 ¿Ya lo tienes claro?

1. ¿Por qué es mala idea rellenar los datos que faltan usando **todas** las fincas antes de separar las del examen?
2. Si llega una finca con una variedad que nunca se vio, ¿qué hace la receta? ¿Te parece razonable? ¿Qué otra cosa podrías hacer?
3. Tu jefe quiere saber «cuánto más rinde la Pastusa» y también «si el modelo servirá el próximo año». ¿Qué cocinero usarías para cada pregunta?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>